<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/ball_trap_transformer_v4_test_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ball-Trap Brain v4 vs Transformer — Test & Size Comparison

This notebook reproduces the TinyGPT architecture from the supplied Transformer notebook, compiles the exact same weights into the pure-discrete Ball-Trap v4 runtime, and compares:

- model parameter counts
- saved FP32 model size
- packed Ball-Trap model size
- output/logit error
- top-1 token agreement
- multiple sequence lengths
- 4/6/8-bit weight representations
- optional trained-checkpoint testing

The default model is deterministic random weights. To test a trained model, put its PyTorch checkpoint path in the configuration cell.

In [ ]:
# If running outside /mnt/data, change these paths.
import os, sys, math, json, tempfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path('/mnt/data')
sys.path.insert(0, str(ROOT))
from ball_trap_transformer_v4 import (
    TinyGPTBallTrapCompiler,
    compare_cases,
    save_ball_program,
    load_ball_program,
    model_bytes,
    packed_theoretical,
)

device = torch.device('cpu')
torch.set_grad_enabled(False)
print('PyTorch:', torch.__version__)
print('Device:', device)

In [ ]:
# TinyGPT definition copied from the supplied Transformer notebook.
vocab = list("0123456789+=") + ["<pad>", "<eos>"]
stoi = {ch:i for i,ch in enumerate(vocab)}
itos = {i:ch for ch,i in stoi.items()}
PAD_ID = stoi['<pad>']
EOS_ID = stoi['<eos>']
max_len = 15

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
    def forward(self, x):
        B,T,C = x.shape
        qkv = self.qkv(x).view(B,T,3,self.n_head,self.d_head).permute(2,0,3,1,4)
        q,k,v = qkv[0],qkv[1],qkv[2]
        att = (q @ k.transpose(-2,-1)) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask, float('-inf'))
        att = F.softmax(att, dim=-1)
        out = att @ v
        out = out.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model,n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model,4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model,d_model),
        )
    def forward(self,x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self,vocab_size,max_len,d_model=64,n_head=4,n_layer=3):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size,d_model)
        self.pos_emb = nn.Embedding(max_len,d_model)
        self.blocks = nn.ModuleList([Block(d_model,n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model,vocab_size)
    def forward(self,idx):
        B,T = idx.shape
        pos = torch.arange(T,device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks: x = block(x)
        x = self.ln_f(x)
        return self.head(x)

SEED = 1234
torch.manual_seed(SEED)
model = TinyGPT(len(vocab), max_len).to(device).eval()
print('Parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
# OPTIONAL: test a trained checkpoint instead of the deterministic random model.
# Set TRAINED_CHECKPOINT to a .pt/.pth file containing a TinyGPT state_dict.
TRAINED_CHECKPOINT = None

if TRAINED_CHECKPOINT:
    ckpt = torch.load(TRAINED_CHECKPOINT, map_location=device)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        ckpt = ckpt['state_dict']
    model.load_state_dict(ckpt)
    model.eval()
    print('Loaded trained checkpoint:', TRAINED_CHECKPOINT)
else:
    print('Using deterministic random weights.')

In [ ]:
# Save the FP32 PyTorch model and report its true file size.
outdir = ROOT / 'ball_trap_test_artifacts'
outdir.mkdir(exist_ok=True)
fp32_path = outdir / 'tinygpt_fp32_state.pt'
torch.save({'model_state_dict': model.state_dict()}, fp32_path)
fp32_bytes = fp32_path.stat().st_size
param_bytes = model_bytes(model)
print(f'Raw parameter bytes (FP32 only): {param_bytes:,} ({param_bytes/1024:.2f} KiB)')
print(f'Saved torch checkpoint file:       {fp32_bytes:,} ({fp32_bytes/1024:.2f} KiB)')

In [ ]:
# Compile the same model at several discrete weight precisions.
brains = {}
compiled_info = {}
for bits in (8,6,4):
    compiler = TinyGPTBallTrapCompiler(weight_bits=bits, activation_bits=16, activation_frac=8, lut_bits=12)
    brain = compiler.compile(model)
    optimize_stats = __import__('ball_trap_transformer_v4').optimize_ball_graph(brain)
    path = outdir / f'tinygpt_balltrap_v4_{bits}bit.btb'
    save_ball_program(brain, path)
    brains[bits] = (compiler, brain)
    compiled_info[bits] = {
        'parameter_count': brain.parameter_count,
        'packed_parameter_bytes': brain.packed_parameter_bytes,
        'btb_file_bytes': path.stat().st_size,
        'theoretical_weight_bytes': packed_theoretical(model,bits),
        'graph_nodes': len(brain.ir.balls),
        'edges': len(brain.ir.edges),
        'ops': len(brain.ir.ops),
    }
compiled_info

In [ ]:
# Exact size comparison.
import pandas as pd
rows = [{
    'model': 'Transformer FP32',
    'weight bits': 32,
    'parameter count': sum(p.numel() for p in model.parameters()),
    'raw parameter bytes': param_bytes,
    'saved file bytes': fp32_bytes,
    'saved file KiB': round(fp32_bytes/1024,2),
}]
for bits in (8,6,4):
    info=compiled_info[bits]
    rows.append({
        'model': f'Ball-Trap v4 {bits}-bit',
        'weight bits': bits,
        'parameter count': info['parameter_count'],
        'raw parameter bytes': info['packed_parameter_bytes'],
        'saved file bytes': info['btb_file_bytes'],
        'saved file KiB': round(info['btb_file_bytes']/1024,2),
    })
size_df=pd.DataFrame(rows)
size_df['file size vs FP32'] = size_df['saved file bytes']/fp32_bytes
size_df['raw weight size vs FP32'] = size_df['raw parameter bytes']/param_bytes
size_df

In [ ]:
# Packing savings if we stored ONLY the quantized weights (no graph/metadata).
for bits in (8,6,4,2,1):
    theo = packed_theoretical(model,bits)
    print(f'{bits}-bit: {theo:,} bytes = {theo/1024:.2f} KiB = {theo/param_bytes:.3%} of FP32 parameter bytes')

In [ ]:
# Helper to encode a string with the notebook vocabulary.
def encode(s):
    return torch.tensor([[stoi[c] for c in s]], dtype=torch.long)

cases = [
    '1+2=',
    '1+2+3=',
    '1+2+19=',
    '1+2+199=',
    '123+=',
    '987654321+=',
    '1+2+19999=',
]
case_tensors = [encode(x) for x in cases]
for s,x in zip(cases,case_tensors):
    print(f'{s:14s} -> shape {tuple(x.shape)}')

In [ ]:
# Compare Transformer vs pure-discrete Ball-Trap outputs.
all_results=[]
for bits,(compiler,brain) in brains.items():
    results=compare_cases(model,compiler,brain,case_tensors)
    for r,s in zip(results,cases):
        r['bits']=bits; r['input']=s
        all_results.append(r)
results_df=pd.DataFrame(all_results)[['bits','input','T','mae','max_error','top1','ref_last','ball_last']]
results_df

In [ ]:
# Default sweep: 3 deterministic random sequences at each length. Increase range(3) for a larger benchmark; pure-Python discrete execution is intentionally slower than PyTorch kernels.
rng=np.random.default_rng(2026)
sweep=[]
for T in [1,4,8,12,15]:
    for case in range(1):
        ids=torch.tensor([[int(x) for x in rng.integers(0,len(vocab)-2,size=T)]],dtype=torch.long)
        ref=model(ids)
        for bits,(compiler,brain) in brains.items():
            out=compiler.run(brain,ids)
            err=(ref-out).abs()
            sweep.append({
                'bits':bits,'T':T,
                'mae':float(err.mean()),
                'max_error':float(err.max()),
                'top1_agreement':float((ref.argmax(-1)==out.argmax(-1)).float().mean()),
            })
sweep_df=pd.DataFrame(sweep).groupby(['bits','T']).mean(numeric_only=True).reset_index()
sweep_df

In [ ]:
# Verify the exported .btb files reload correctly and preserve discrete outputs exactly.
reload_rows=[]
for bits,(compiler,brain) in brains.items():
    path=outdir/f'tinygpt_balltrap_v4_{bits}bit.btb'
    loaded=load_ball_program(path)
    x=encode('1+2+199=')
    y1=compiler.run(brain,x)
    y2=compiler.run(loaded and loaded and loaded, x) if False else None
    # load_ball_program returns a brain object; instantiate the matching compiler.
    compiler2=TinyGPTBallTrapCompiler(weight_bits=bits,activation_bits=16,activation_frac=8,lut_bits=12)
    a=compiler2.run(loaded,x)
    reload_rows.append({'bits':bits,'max_reload_difference':float((y1-a).abs().max()),'file_bytes':path.stat().st_size})
pd.DataFrame(reload_rows)

In [ ]:
# Inspect the ball-trap program itself.
for bits,(compiler,brain) in brains.items():
    print(f'\n=== {bits}-bit brain ===')
    print(brain.report())

## Interpreting the size numbers

There are three different sizes worth keeping separate:

1. **FP32 Transformer parameters** — the actual learned tensors at 32 bits/value.
2. **Packed Ball-Trap parameters** — the same learned tensors quantized and bit-packed.
3. **`.btb` Ball-Trap program file** — packed parameters **plus** the Ball-Trap graph, operation records, metadata, and serialization information.

The `.btb` file can therefore be larger than the theoretical packed weights. The eventual optimization target is to reduce both the parameter payload and the graph representation, including weight sharing and reusable circuits.

In [ ]:
# Optional: test a particular input and inspect the last-position top tokens.
def show_prediction(text,bits=8,topk=5):
    ids=encode(text)
    ref=model(ids)
    compiler,brain=brains[bits]
    out=compiler.run(brain,ids)
    for name,logits in [('Transformer',ref),('Ball-Trap',out)]:
        vals,idxs=torch.topk(logits[0,-1],topk)
        toks=[itos[int(i)] for i in idxs]
        print(name, list(zip(toks,[round(float(v),5) for v in vals])))
    print('MAE:',float((ref-out).abs().mean()))
    print('Same top token:',bool(ref.argmax(-1)[0,-1]==out.argmax(-1)[0,-1]))

show_prediction('1+2+199=',bits=8)

## Testing a trained model

Set `TRAINED_CHECKPOINT` near the top to the path of your saved TinyGPT `state_dict`, rerun the model/compile cells, and then rerun the comparison cells.

For a trained arithmetic model, the most informative experiment is to compare **task accuracy as well as logit agreement**. This notebook intentionally does not call the Ball-Trap result “equivalent” merely from low numerical error: we should verify that it preserves the actual learned behavior on the held-out arithmetic examples.